In [43]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, Window
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.tuning import ParamGridBuilder
from pyspark.ml import Pipeline, PipelineModel
import matplotlib.pyplot as plt
import numpy as np

In [30]:
spark = SparkSession.builder \
    .appName("Segmentacion_Perfilado_Clientes") \
    .getOrCreate()

clientes_df = spark.read.parquet("/home/jovyan/work/data/CLIENTS_CLEAN", header=True, inferSchema=True)
behavioural_df = spark.read.parquet("/home/jovyan/work/data/BEHAVIOURAL_CLEAN", header=True, inferSchema=True)

In [31]:
clientes_df.columns

['CLIENT_ID',
 'NON_COMPLIANT_CONTRACT',
 'NAME_PRODUCT_TYPE',
 'GENDER',
 'TOTAL_INCOME',
 'AMOUNT_PRODUCT',
 'INSTALLMENT',
 'EDUCATION',
 'MARITAL_STATUS',
 'HOME_SITUATION',
 'REGION_SCORE',
 'AGE_IN_YEARS',
 'JOB_SENIORITY',
 'HOME_SENIORITY',
 'LAST_UPDATE',
 'OWN_INSURANCE_CAR',
 'FAMILY_SIZE',
 'PROACTIVE_SCORING',
 'BEHAVIORAL_SCORING',
 'DAYS_LAST_INFO_CHANGE',
 'NUMBER_OF_PRODUCTS',
 'OCCUPATION',
 'DIGITAL_CLIENT',
 'HOME_OWNER',
 'EMPLOYER_ORGANIZATION_TYPE',
 'NUM_PREVIOUS_LOAN_APP',
 'LOAN_ANNUITY_PAYMENT_MAX',
 'LOAN_ANNUITY_PAYMENT_MIN',
 'LOAN_ANNUITY_PAYMENT_SUM',
 'LOAN_APPLICATION_AMOUNT_MAX',
 'LOAN_APPLICATION_AMOUNT_MIN',
 'LOAN_APPLICATION_AMOUNT_SUM',
 'LOAN_CREDIT_GRANTED_MAX',
 'LOAN_CREDIT_GRANTED_MIN',
 'LOAN_CREDIT_GRANTED_SUM',
 'LOAN_VARIABLE_RATE_MAX',
 'LOAN_VARIABLE_RATE_MIN',
 'NUM_STATUS_ANNULLED',
 'NUM_STATUS_AUTHORIZED',
 'NUM_STATUS_DENIED',
 'NUM_STATUS_NOT_USED',
 'NUM_FLAG_INSURED']

In [ ]:
# df_id_unido = clientes_df.alias("c").join(
#     behavioural_df.alias("b"),
#     on="CLIENT_ID",
#     how="inner"
# )

In [ ]:
# df_id_unido.show(1)

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+-----------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----------------

In [ ]:
# len(df_id_unido.columns)

58

In [ ]:
# NUMERICAL_COLS = [
#     "JOB_SENIORITY",
#     "INSTALLMENT",
#     "FAMILY_SIZE",
#     "PROACTIVE_SCORING",
#     "BEHAVIORAL_SCORING",
#     "DAYS_LAST_INFO_CHANGE",
#     "NUMBER_OF_PRODUCTS",
#     "TOTAL_INCOME",
#     "AMOUNT_PRODUCT",
#     "INSTALLMENT",
#     "REGION_SCORE",
# ]

# CATEGORICAL_COLS = [
# ]

In [ ]:
# df_imputado = df_id_unido.na.fill(0, NUMERICAL_COLS)
# df_imputado = df_imputado.na.fill("NA_MISSING", subset=CATEGORICAL_COLS)

In [ ]:
# df_imputado.show(5)

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+-----------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----------------

In [ ]:
# #Todas las columnas ya están limpias y sin valores nulos

# all_columns = NUMERICAL_COLS + CATEGORICAL_COLS

In [ ]:
# indexers = [
#     StringIndexer(inputCol=c, outputCol=c + "_Index", handleInvalid="keep")
#     for c in CATEGORICAL_COLS
# ]

# encoders = [
#     OneHotEncoder(inputCols=[c + "_Index"], outputCols=[c + "_OHE"], dropLast=True)
#     for c in CATEGORICAL_COLS
# ]

In [ ]:
# indexers

[]

In [58]:
# assembler = VectorAssembler(
#     inputCols=all_columns,
#     outputCol="unscaled_features" # Nombre temporal antes de escalar
# )


# scaler = StandardScaler(
#     inputCol="unscaled_features",
#     outputCol="features", # ESTA es tu columna final
#     withStd=True,
#     withMean=False # No centrar para datos dispersos (OHE), si tienes muchos zeros
#)

In [29]:
# PCA_COMPONENTS = 10 
# pca = PCA(k=PCA_COMPONENTS, inputCol="features", outputCol="pca_features")

In [ ]:
# pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, pca])

# pipeline_model = pipeline.fit(df_imputado)

# df_features = pipeline_model.transform(df_imputado)

# df_features.select("CLIENT_ID", "features").show(5, False)

# print(f"Dimensiones totales del vector 'features': {len(all_columns)}")

+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|CLIENT_ID   |features                                                                                                                                                                                                         |
+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ES182147947X|[0.29233387252426585,2.614051124979187,2.281310270135329,3.8095503356063487,0.0,2.5873136648080646,0.0,1.6056954949819635,2.7444960406161614,2.614051124979187,2.7595859054723357]                               |
|ES182389511E|[0.05498566749779831,1.0976702460102052,2.281310270135329,3.735953097072094,1.99944595

In [3]:
columnas_beh = [
    "CREDICT_CARD_BALANCE",
    "CREDIT_CARD_LIMIT",
    "CREDIT_CARD_DRAWINGS_ATM",
    "CREDIT_CARD_DRAWINGS_POS",
    "CREDIT_CARD_DRAWINGS_OTHER",
    "CREDIT_CARD_DRAWINGS",
    "CREDIT_CARD_PAYMENT",
    "NUMBER_DRAWINGS_ATM",
    "NUMBER_DRAWINGS",
    "NUMBER_INSTALMENTS"
]

columnas_base = ["CLIENT_ID"] + columnas_beh

In [ ]:
#  comprobación de que las columnas funcionan
#  for col in columnas_beh:
#     behavioural_df.select(col).distinct().show(5)

In [4]:
assembler = VectorAssembler(
    inputCols=columnas_beh,
    outputCol="features"
)

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=False
)

In [5]:
feature_pipeline = Pipeline(stages=[assembler, scaler])

behavioural_df_num = behavioural_df.select(columnas_base)

df_features = feature_pipeline.fit(behavioural_df_num).transform(behavioural_df_num)

# VAMOS CON EL FOKIN K-MEANS

In [37]:
FEATURES_COL = "scaled_features" 
K_range = np.arange(2, 4).tolist()
random_seed = 777

inertias = []
silhouettes = []

# Definir el estimador K-Means
kmeans = KMeans(featuresCol=FEATURES_COL, seed=random_seed)

# Definir el evaluador de Silhouette (métrica de calidad de clustering)
evaluator = ClusteringEvaluator(
    featuresCol=FEATURES_COL, 
    predictionCol="prediction", 
    metricName="silhouette", 
    distanceMeasure="squaredEuclidean"
)

# Definir la grilla de parámetros
paramGrid = (ParamGridBuilder()
    .addGrid(kmeans.k, K_range)
    .build()
)

print(f"Iniciando Grid Search distribuido sobre K: {K_range}...")

# --- 2. BÚSQUEDA DISTRIBUIDA ---
for params in paramGrid:
    # Entrenar el modelo con el K actual
    model = kmeans.fit(df_features, params)
    
    # Calcular WCSS (Inercia)
    wcss = model.summary.trainingCost
    inertias.append(wcss)
    
    # Transformar y calcular Silhouette Score
    df_predictions = model.transform(df_features)
    sil_score = evaluator.evaluate(df_predictions)
    silhouettes.append(sil_score)
    
    k = params.get(kmeans.k)
    print(f"K={k}: WCSS (Inercia) = {wcss:.2f}, Silhouette Score = {sil_score:.4f}")

# --- 3. GUARDAR RESULTADOS ---
# Crear un DataFrame con los resultados
df_results = spark.createDataFrame(
    zip(K_range, inertias, silhouettes), 
    ["k", "Inertia", "Silhouette"]
)

# Guardar los resultados en un CSV pequeño (¡ahora sí se puede hacer .collect()!)
df_results.toPandas().to_csv("kmeans_metrics_results.csv", index=False)
print("\n✅ Cálculo distribuido completado.")
print("El resultado para plotear se ha guardado en: 'kmeans_metrics_results.csv'")

Iniciando Grid Search distribuido sobre K: [2, 3]...
K=2: WCSS (Inercia) = 16002389.95, Silhouette Score = 0.9888
K=3: WCSS (Inercia) = 12265638.47, Silhouette Score = 0.8935

✅ Cálculo distribuido completado.
El resultado para plotear se ha guardado en: 'kmeans_metrics_results.csv'


In [38]:
# ASUMIMOS que df_features ahora contiene CLIENT_ID y scaled_features

K_OPTIMO = 2 

kmeans_final = KMeans(
    featuresCol="scaled_features", 
    predictionCol="segmento_cliente", 
    k=K_OPTIMO,
    seed=42
)
model_final = kmeans_final.fit(df_features)

# La transformación simplemente añade la columna 'segmento_cliente' a df_features
df_segmentado = model_final.transform(df_features)

# Eliminamos la línea incorrecta de withColumn
# df_segmentado = df_segmentado.withColumn(...) <--- ELIMINAR ESTO

print(f"Modelo K-Means entrenado con K={K_OPTIMO}. Segmentos asignados.")
# La selección ahora funcionará porque CLIENT_ID está en el DataFrame desde el inicio
df_segmentado.select("CLIENT_ID", "segmento_cliente").show(5)


Modelo K-Means entrenado con K=2. Segmentos asignados.
+------------+----------------+
|   CLIENT_ID|segmento_cliente|
+------------+----------------+
|ES182147947X|               0|
|ES182389511E|               0|
|ES182423955L|               0|
|ES182265271Q|               0|
|ES182332566J|               0|
+------------+----------------+
only showing top 5 rows



In [39]:
# 1. Calcular el promedio de las variables clave por segmento
# Agrupamos por el resultado del clustering ('segmento_cliente')
df_perfil_numerico_beh = df_segmentado.groupBy("segmento_cliente").agg(
    *[F.mean(col).alias(f"MEAN_{col}") for col in columnas_beh]
).orderBy("segmento_cliente")

# 2. Mostrar la tabla de perfiles
# Esto te permitirá comparar las medias de los dos segmentos (0 y 1)
df_perfil_numerico_beh.show(truncate=False)


+----------------+-------------------------+----------------------+-----------------------------+-----------------------------+-------------------------------+-------------------------+------------------------+------------------------+--------------------+-----------------------+
|segmento_cliente|MEAN_CREDICT_CARD_BALANCE|MEAN_CREDIT_CARD_LIMIT|MEAN_CREDIT_CARD_DRAWINGS_ATM|MEAN_CREDIT_CARD_DRAWINGS_POS|MEAN_CREDIT_CARD_DRAWINGS_OTHER|MEAN_CREDIT_CARD_DRAWINGS|MEAN_CREDIT_CARD_PAYMENT|MEAN_NUMBER_DRAWINGS_ATM|MEAN_NUMBER_DRAWINGS|MEAN_NUMBER_INSTALMENTS|
+----------------+-------------------------+----------------------+-----------------------------+-----------------------------+-------------------------------+-------------------------+------------------------+------------------------+--------------------+-----------------------+
|0               |623.2155948311471        |1717.085803437802     |16.63747472907447            |8.856158468354822            |0.9824616722535193            

# Segmento 0:	
Cliente Inactivo / "Dormido"	Tiene tarjeta, pero su uso es casi nulo. El límite y saldo son bajos. Podrían ser clientes de bajo riesgo, pero con bajo potencial de ingresos para el banco.

# Segmento 1:	
Cliente Activo / "Premium" / Alto Riesgo	Alta utilización de la tarjeta, alta deuda, altos límites y altos pagos. Es el principal motor de ingresos por intereses y comisiones, pero también podría ser el principal motor de riesgo de crédito.

## 1. Comprender la Base de Clientes (Perfilado)
Logrado: Tienes un perfil dual: el cliente que usa la tarjeta activamente (S1) y el cliente inactivo (S0). Ahora puedes añadir variables demográficas (edad, ingresos) a este perfil para decir: "El S1 es un cliente activo, joven y de altos ingresos".

## 2. Identificar Patrones de Comportamiento Clave
Logrado: El patrón clave es el uso intensivo de crédito. El patrón de comportamiento en S1 es de alto consumo y alta deuda rotatoria, mientras que el patrón en S0 es de inactividad total.

## 3. Detectar Riesgos de Abandono o Pérdida de Rentabilidad
Riesgo de Abandono (S0): El Segmento 0 es altamente vulnerable al churn. Si no usan el producto, es fácil que cambien de banco o cancelen. Oportunidad: Crear campañas de engagement.

Riesgo de Pérdida de Rentabilidad/Riesgo Crediticio (S1): El Segmento 1 es altamente rentable, pero su alta deuda ($2,753 de saldo promedio) y altos reintegros en cajeros (DRAWINGS_ATM) sugieren que están cerca de su límite. Riesgo: Si pierden ingresos, podrían entrar en impago.

## 4. Encontrar Oportunidades de Crecimiento
Oportunidad (S0): Los clientes inactivos son un objetivo fácil para cross-selling de productos complementarios o para campañas de activación (ej. ofertas de cashback para la primera compra).

Oportunidad (S1): Dado su alto nivel de confianza en el banco (altos límites), pueden ser candidatos para préstamos personales de mayor valor o productos de inversión.

## 5. Proponer Indicadores (KPIs)
KPI Primario para S0 (Inactivos): Tasa de Activación Mensual (porcentaje de S0 que realiza una compra o retiro).

KPI Primario para S1 (Activos): Tasa de Utilización del Límite (MEAN_CREDIT_CARD_BALANCE / MEAN_CREDIT_CARD_LIMIT). Si es muy alta (cerca del 80-90%), el cliente está cerca del sobreendeudamiento, lo que activa una señal de riesgo.

In [40]:
columnas_cli = ['CLIENT_ID',
 'NON_COMPLIANT_CONTRACT',
 'NAME_PRODUCT_TYPE',
 'GENDER',
 'TOTAL_INCOME',
 'AMOUNT_PRODUCT',
 'INSTALLMENT',
 'EDUCATION',
 'MARITAL_STATUS',
 'HOME_SITUATION',
 'REGION_SCORE',
 'AGE_IN_YEARS',
 'JOB_SENIORITY',
 'HOME_SENIORITY',
 'LAST_UPDATE',
 'OWN_INSURANCE_CAR',
 'FAMILY_SIZE',
 'PROACTIVE_SCORING',
 'BEHAVIORAL_SCORING',
 'DAYS_LAST_INFO_CHANGE',
 'NUMBER_OF_PRODUCTS',
 'OCCUPATION',
 'DIGITAL_CLIENT',
 'HOME_OWNER',
 'EMPLOYER_ORGANIZATION_TYPE',
 'NUM_PREVIOUS_LOAN_APP',
 'LOAN_ANNUITY_PAYMENT_MAX',
 'LOAN_ANNUITY_PAYMENT_MIN',
 'LOAN_ANNUITY_PAYMENT_SUM',
 'LOAN_APPLICATION_AMOUNT_MAX',
 'LOAN_APPLICATION_AMOUNT_MIN',
 'LOAN_APPLICATION_AMOUNT_SUM',
 'LOAN_CREDIT_GRANTED_MAX',
 'LOAN_CREDIT_GRANTED_MIN',
 'LOAN_CREDIT_GRANTED_SUM',
 'LOAN_VARIABLE_RATE_MAX',
 'LOAN_VARIABLE_RATE_MIN',
 'NUM_STATUS_ANNULLED',
 'NUM_STATUS_AUTHORIZED',
 'NUM_STATUS_DENIED',
 'NUM_STATUS_NOT_USED',
 'NUM_FLAG_INSURED']

df_segmentado_clientes = df_segmentado.select("CLIENT_ID", "segmento_cliente").join(
    clientes_df.select(columnas_cli),
    on="CLIENT_ID",
    how="left"
)

In [41]:
df_perfil_numerico_cli_num = df_segmentado_clientes.groupBy("segmento_cliente").agg(
    *[F.mean(col).alias(f"MEAN_{col}") for col in columnas_cli if col != "CLIENT_ID"]
).orderBy("segmento_cliente")

df_perfil_numerico_cli_num.show(truncate=False)

df_perfil_numerico_cli_cat = df_segmentado_clientes.groupBy("segmento_cliente").agg(
    *[F.mode(col).alias(f"MODE_{col}") for col in columnas_cli if col != "CLIENT_ID"]
).orderBy("segmento_cliente")

df_perfil_numerico_cli_cat.show(truncate=False)

+----------------+---------------------------+----------------------+-----------+------------------+-------------------+------------------+--------------+-------------------+-------------------+--------------------+------------------+------------------+-------------------+------------------+----------------------+------------------+----------------------+-----------------------+--------------------------+-----------------------+---------------+-------------------+---------------+-------------------------------+--------------------------+-----------------------------+-----------------------------+-----------------------------+--------------------------------+--------------------------------+--------------------------------+----------------------------+----------------------------+----------------------------+---------------------------+---------------------------+------------------------+--------------------------+----------------------+------------------------+---------------------+
|s